In [ ]:
import yfinance as yf
import pandas as pd
import networkx as nx
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster#, cophenet
from scipy.spatial.distance import pdist
from sklearn.preprocessing import StandardScaler
from community import community_louvain
import matplotlib.pyplot as plt

In [ ]:
date1 = '2025-01-01'
date2 = '2026-01-01'

stocks = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'TSLA', 'BRK-B', 'JNJ', 'V', 'WMT',
'NVDA','JPM','UNH','HD','PG','MA','DIS','PYPL','ADBE','NFLX',
'PEP','INTC','KO','NKE','MRK','PFE','T','VZ','XOM','CSCO']
data = yf.download(stocks, start=date1, end=date2,auto_adjust=False)['Adj Close']

In [ ]:
returns = data.pct_change().dropna()
scaler = StandardScaler()
normalized_returns = pd.DataFrame(scaler.fit_transform(returns), columns=returns.columns, index=returns.index)
print(normalized_returns)

In [ ]:
correlation_matrix = normalized_returns.corr()
print(correlation_matrix)

In [ ]:
G = nx.Graph()
G.add_nodes_from(stocks)
threshold = 0.5

for i in range(len(stocks)):
    for j in range(i + 1,len(stocks)):
        corr = correlation_matrix.iloc[i, j]
        if abs(corr) > threshold:
            G.add_edge(stocks[i], stocks[j], weight=corr)

In [ ]:
mst = nx.minimum_spanning_tree(G)
plt.figure(figsize=(10, 10))
pos = nx.spring_layout(mst)
nx.draw(mst, pos, with_labels=True, node_size=1000, node_color="skyblue", edge_color="gray")
plt.title("Minimum Spanning Tree of Stocks")
plt.show()

In [ ]:
partition = community_louvain.best_partition(G)
print("Community detection using Louvain method:")
print(partition)

In [ ]:
df = pd.DataFrame(list(partition.items()), columns=["ticker", "label"])
groups = df.groupby("label")["ticker"].apply(list).to_dict()
groups

In [ ]:
corr_matrix = correlation_matrix
distance_matrix = 1 - corr_matrix.abs()
condensed_distance_matrix = pdist(distance_matrix)
linkage_matrix = linkage(condensed_distance_matrix, method='ward')
plt.figure(figsize=(10, 7))
dendrogram(linkage_matrix, labels=stocks)
plt.axhline(y=2.5, color="red", linestyle="--", linewidth=2)
plt.xlabel("Stock")
plt.ylabel("Distance")
plt.plot([],[])
plt.show()

In [ ]:
benchmark_data = yf.download("SPY", start=date1, end=date2,auto_adjust=False)['Adj Close']
benchmark_returns = benchmark_data.pct_change().dropna()

In [ ]:
methods = ['single', 'complete', 'average', 'centroid', 'ward', 'median']
max_d_values = [0.3,0.4,0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.2, 1.4]

In [ ]:
fig, axes = plt.subplots(len(methods), len(max_d_values), figsize=(40, 20), sharex=True, sharey=True)
fig.suptitle("Cluster-Based Portfolio vs Benchmark", fontsize=20)

distance_matrix = 1 - correlation_matrix.abs()
condensed_distance_matrix = pdist(distance_matrix)

for i, method in enumerate(methods):
    linkage_matrix = linkage(condensed_distance_matrix, method=method)
    for j, max_d in enumerate(max_d_values):
        clusters = fcluster(linkage_matrix, max_d, criterion='distance')

        # Cluster-based portfolio
        cluster_returns = []
        unique_clusters = np.unique(clusters)
        for cluster in unique_clusters:
            cluster_stocks = [stocks[k] for k in range(len(stocks)) if clusters[k] == cluster]
            cluster_data = returns[cluster_stocks].mean(axis=1)
            cluster_returns.append(cluster_data)
            #print(method, max_d,cluster,cluster_stocks)

        # Combine cluster returns into a portfolio
        combined_returns = pd.concat(cluster_returns, axis=1).mean(axis=1)
        combined_cum_returns = (1 + combined_returns).cumprod() - 1
        benchmark_cum_returns = (1 + benchmark_returns).cumprod() - 1

        # Plot
        ax = axes[i, j]
        combined_cum_returns.plot(ax=ax, label="Cluster-Based Portfolio", color="blue")
        benchmark_cum_returns.plot(ax=ax, label="S&P 500 ETF (SPY)", color="orange")
        ax.set_title(f"{method}, max_d={max_d}")
        ax.set_xlabel("Date")
        ax.set_ylabel("Cumulative Returns")
        ax.legend()

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()